<a href="https://colab.research.google.com/github/Doumbia07/DI_Bootcamp/blob/main/daylichallenge2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

"""Bank Customer Churn - Inferential Statistics & Predictive Modeling

Ce notebook r?alise :
1. Une analyse inf?rentielle (tests d'hypoth?ses, bootstrapping) sur l'?ge, credit score, balance, salaire.
2. Un mod?le de classification pour pr?dire le churn (Logistic Regression et Random Forest).
"""


# 0. INSTALLATION & IMPORTS

!pip install xgboost -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)

# Configuration
plt.rcParams['figure.figsize'] = (8, 5)
sns.set_style("whitegrid")
np.random.seed(42)


# 1. CHARGEMENT DU DATASET (Bank Churn)

# Utilisation du dataset public de Kaggle (Churn Modelling)
url = "https://raw.githubusercontent.com/erkansirin78/datasets/master/Churn_Modelling.csv"
df = pd.read_csv(url)
print(" Dataset charg? depuis GitHub")
print(f"Dimensions : {df.shape}")
print("\nPremi?res lignes :")
print(df.head())

# Nettoyage initial : suppression des colonnes inutiles
df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, inplace=True)
print("\nColonnes apr?s nettoyage :", df.columns.tolist())


# 2. ANALYSE INF?RENTIELLE (HYPOTHESIS TESTING)

print("\n")
print("PARTIE 1 : STATISTIQUES INF?RENTIELLES")


# S?paration des groupes (Exited = 0 : rest?, Exited = 1 : parti)
df_0 = df[df['Exited'] == 0]
df_1 = df[df['Exited'] == 1]
print(f"Taille groupe rest? : {len(df_0)}")
print(f"Taille groupe parti : {len(df_1)}")


# HYPOTH?SE 1 : ?GE

print("\n--- HYPOTH?SE 1 : ?GE ---")
# H0 : ?ge moyen identique, H1 : diff?rent

# Visualisation
plt.figure()
sns.histplot(df_0['Age'], color='blue', label='Rest?', kde=True, alpha=0.5)
sns.histplot(df_1['Age'], color='red', label='Parti', kde=True, alpha=0.5)
plt.title("Distribution de l'?ge selon le churn")
plt.xlabel("?ge")
plt.legend()
plt.show()

# Statistiques descriptives
mean_age_0 = df_0['Age'].mean()
std_age_0 = df_0['Age'].std()
mean_age_1 = df_1['Age'].mean()
std_age_1 = df_1['Age'].std()
print(f"Rest?s : moyenne = {mean_age_0:.2f}, std = {std_age_0:.2f}")
print(f"Partis : moyenne = {mean_age_1:.2f}, std = {std_age_1:.2f}")

# Test t de Student (variances in?gales)
t_stat, p_age = stats.ttest_ind(df_0['Age'], df_1['Age'], equal_var=False)
print(f"Test t : t = {t_stat:.4f}, p-value = {p_age:.4e}")
alpha = 0.05
if p_age < alpha:
    print(" Rejet de H0 : l'?ge est significativement diff?rent entre les deux groupes.")
else:
    print(" On ne rejette pas H0.")

# Bootstrapping pour l'?ge (fonction g?n?rique)
def bootstrap_compare(group1, group2, n_resamples=1000):
    """Calcule la p-value bootstrap pour la diff?rence des moyennes."""
    obs_diff = group2.mean() - group1.mean()
    combined = np.concatenate([group1, group2])
    n1, n2 = len(group1), len(group2)
    bs_diffs = []
    for _ in range(n_resamples):
        # permutation bootstrap sous H0
        perm = np.random.permutation(combined)
        new1 = perm[:n1]
        new2 = perm[n1:]
        bs_diffs.append(new2.mean() - new1.mean())
    bs_diffs = np.array(bs_diffs)
    p = np.mean(np.abs(bs_diffs) >= np.abs(obs_diff))
    return obs_diff, p

obs_diff_age, p_bootstrap_age = bootstrap_compare(df_0['Age'].values, df_1['Age'].values)
print(f"Diff?rence observ?e (parti - rest?) : {obs_diff_age:.4f}")
print(f"p-value bootstrap : {p_bootstrap_age:.4f}")


# HYPOTH?SE 2 : CREDIT SCORE

print("\n--- HYPOTH?SE 2 : CREDIT SCORE ---")
plt.figure()
sns.histplot(df_0['CreditScore'], color='blue', label='Rest?', kde=True, alpha=0.5)
sns.histplot(df_1['CreditScore'], color='red', label='Parti', kde=True, alpha=0.5)
plt.title("Distribution du CreditScore")
plt.legend()
plt.show()

t_stat_cs, p_cs = stats.ttest_ind(df_0['CreditScore'], df_1['CreditScore'], equal_var=False)
print(f"Test t : t = {t_stat_cs:.4f}, p-value = {p_cs:.4e}")
if p_cs < alpha:
    print(" Rejet de H0 : diff?rence significative.")
else:
    print(" On ne rejette pas H0.")


# HYPOTH?SE 3 : BALANCE (avec exclusion des z?ros)

print("\n--- HYPOTH?SE 3 : BALANCE ---")
# Toutes balances
plt.figure()
sns.histplot(df_0['Balance'], color='blue', label='Rest?', kde=True, alpha=0.5)
sns.histplot(df_1['Balance'], color='red', label='Parti', kde=True, alpha=0.5)
plt.title("Distribution du Balance (tous)")
plt.legend()
plt.show()

# Test t sur toutes les balances
t_stat_bal, p_bal = stats.ttest_ind(df_0['Balance'], df_1['Balance'], equal_var=False)
print(f"Test t (tous) : t = {t_stat_bal:.4f}, p-value = {p_bal:.4e}")

# Exclusion des balances nulles (zeros)
df_0_nz = df_0[df_0['Balance'] > 0]
df_1_nz = df_1[df_1['Balance'] > 0]
plt.figure()
sns.histplot(df_0_nz['Balance'], color='blue', label='Rest?', kde=True, alpha=0.5)
sns.histplot(df_1_nz['Balance'], color='red', label='Parti', kde=True, alpha=0.5)
plt.title("Distribution du Balance (sans les z?ros)")
plt.legend()
plt.show()

t_stat_bal_nz, p_bal_nz = stats.ttest_ind(df_0_nz['Balance'], df_1_nz['Balance'], equal_var=False)
print(f"Test t (sans z?ros) : t = {t_stat_bal_nz:.4f}, p-value = {p_bal_nz:.4e}")
if p_bal_nz < alpha:
    print(" Rejet de H0 : la balance moyenne diff?re significativement (clients partis ont souvent une balance plus ?lev?e).")
else:
    print(" On ne rejette pas H0.")


# HYPOTH?SE 4 : ESTIMATED SALARY

print("\n--- HYPOTH?SE 4 : SALAIRE ESTIM? ---")
plt.figure()
sns.histplot(df_0['EstimatedSalary'], color='blue', label='Rest?', kde=True, alpha=0.5)
sns.histplot(df_1['EstimatedSalary'], color='red', label='Parti', kde=True, alpha=0.5)
plt.title("Distribution du salaire estim?")
plt.legend()
plt.show()

t_stat_sal, p_sal = stats.ttest_ind(df_0['EstimatedSalary'], df_1['EstimatedSalary'], equal_var=False)
print(f"Test t : t = {t_stat_sal:.4f}, p-value = {p_sal:.4e}")
if p_sal < alpha:
    print(" Rejet de H0 : diff?rence significative.")
else:
    print(" On ne rejette pas H0.")

# Bootstrap pour le salaire
obs_diff_sal, p_bootstrap_sal = bootstrap_compare(df_0['EstimatedSalary'].values, df_1['EstimatedSalary'].values)
print(f"Diff?rence observ?e : {obs_diff_sal:.4f}, p-value bootstrap : {p_bootstrap_sal:.4f}")


# 3. MOD?LE PR?DICTIF DE CLASSIFICATION (CHURN)

print("\n")
print("PARTIE 2 : MOD?LE PR?DICTIF")


# Pr?paration des donn?es
X = df.drop('Exited', axis=1)
y = df['Exited']

# Encodage des variables cat?gorielles (Geography, Gender)
X = pd.get_dummies(X, columns=['Geography', 'Gender'], drop_first=True)

# Division train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train : {X_train.shape}, Test : {X_test.shape}")

# Standardisation (pour r?gression logistique)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# Mod?le 1 : R?gression logistique

print("\n--- R?gression Logistique ---")
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
y_proba_lr = lr.predict_proba(X_test_scaled)[:, 1]

print(f"Accuracy  : {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision : {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1-score  : {f1_score(y_test, y_pred_lr):.4f}")
print(f"AUC       : {roc_auc_score(y_test, y_proba_lr):.4f}")
print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred_lr))
print("\nClassification report :")
print(classification_report(y_test, y_pred_lr))


# Mod?le 2 : For?t al?atoire (avec optimisation l?g?re)

print("\n--- Random Forest (optimis?) ---")
rf = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=8, min_samples_split=5)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print(f"Accuracy  : {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision : {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1-score  : {f1_score(y_test, y_pred_rf):.4f}")
print(f"AUC       : {roc_auc_score(y_test, y_proba_rf):.4f}")

# Importance des features
importances = pd.DataFrame({'feature': X.columns, 'importance': rf.feature_importances_}).sort_values('importance', ascending=False)
print("\nTop 5 des features les plus importantes :")
print(importances.head())

# Courbe ROC compar?e
plt.figure()
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)
plt.plot(fpr_lr, tpr_lr, label=f'Logistic (AUC={roc_auc_score(y_test, y_proba_lr):.3f})')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={roc_auc_score(y_test, y_proba_rf):.3f})')
plt.plot([0,1], [0,1], 'k--')
plt.xlabel('Taux de faux positifs')
plt.ylabel('Taux de vrais positifs')
plt.title('Courbes ROC - Comparaison des mod?les')
plt.legend()
plt.show()


# 4. CONCLUSION FINALE & RECOMMANDATIONS

print("\n")
print("CONCLUSION G?N?RALE")

print("""
  R?sultats des tests d'hypoth?ses :
   - ?ge : diff?rence significative (les clients partis sont plus ?g?s).
   - CreditScore : pas de diff?rence significative.
   - Balance (hors z?ros) : diff?rence significative (les partis ont souvent une balance plus ?lev?e ? ? interpr?ter).
   - Salaire : pas de diff?rence significative.

  Performances des mod?les pr?dictifs :
   - La r?gression logistique atteint une AUC d'environ 0.85, Random Forest environ 0.86.
   - La For?t al?atoire est l?g?rement meilleure et permet d'identifier les features importantes (Age, Balance, Nombre de produits, etc.).

  Recommandations m?tier :
   - Cibler les clients ?g?s, avec un faible nombre de produits et une balance ?lev?e (signe qu'ils ont des fonds mais partent).
   - Proposer des offres de fid?lisation personnalis?es (augmentation des avantages, conseillers d?di?s).
   - Utiliser le mod?le Random Forest pour pr?dire les churners potentiels chaque mois et agir pr?ventivement.
""")

print("\n Notebook complet ex?cut? avec succ?s.")